In [2]:
import pandas as pd

In [3]:
df_satisfacao = pd.read_csv('satisfacao_clientes_expandido.csv')
df_satisfacao.head(10)

df_reserva = pd.read_csv('reservas_viagens_expandido.csv')
df_reserva.head(10)

,id_reserva,data_reserva,id_cliente,roteiro,modalidade,n_pessoas,mes_viagem,valor_total_R$,custo_operacional_R$,canal_origem,status
0,RES000001,2022-08-04,CLI008544,Caminho Primitivo,Autoguiado,4,Agosto,46769,27592,WhatsApp,Concluída
1,RES000002,2021-09-18,CLI015932,Caminho do Norte,Premium,4,Setembro,62595,45282,WhatsApp,Cancelada
2,RES000003,2022-08-01,CLI023473,Caminho do Norte,Premium,3,Agosto,44761,34282,YouTube,Concluída
3,RES000004,2020-09-08,CLI008053,Caminho Português,Premium,1,Setembro,13292,7896,Google,Concluída
4,RES000005,2020-04-06,CLI006063,Caminho do Norte,Premium,1,Abril,17843,13074,YouTube,Concluída
5,RES000006,2026-10-31,CLI019962,Via Francigena,Guiado,2,Outubro,25461,15173,Facebook,Concluída
6,RES000007,2026-01-29,CLI016327,Via Francigena,Premium,5,Janeiro,78306,59866,YouTube,Concluída
7,RES000008,2020-12-27,CLI027382,Caminho Português,Autoguiado,1,Dezembro,13563,9838,TikTok,Concluída
8,RES000009,2023-07-01,CLI001148,Caminho do Norte,Autoguiado,1,Julho,14731,9894,Instagram,Concluída
9,RES000010,2021-01-15,CLI025564,Caminho Português,Premium,4,Janeiro,50393,37402,Indicação,Cancelada


In [4]:
# ---Juntar as bases (Merge) e Criar a Matriz para o KNN ---

# 1. Trazer a coluna 'roteiro' da base de reservas para a base de satisfação
# O Pandas olha o 'id_reserva' em ambas e traz o roteiro correspondente
df_completo = pd.merge(
    df_satisfacao,
    df_reserva[['id_reserva', 'roteiro']],
    on='id_reserva',
    how='left'
)

print("Bases combinadas com sucesso!")
print("Amostra dos dados combinados (agora com a coluna roteiro):")
display(df_completo[['id_cliente', 'id_reserva', 'roteiro', 'nota_roteiro']].head(5))
print("-" * 50)

# 2. Transformar a tabela combinada no formato de Matriz (Pivot Table)
matriz_recomendacao = df_completo.pivot_table(
    index='id_cliente',
    columns='roteiro',
    values='nota_roteiro'
)

print(f"Matriz de Recomendação criada:")
print(f"Total de Clientes (Linhas): {matriz_recomendacao.shape[0]}")
print(f"Total de Roteiros (Colunas): {matriz_recomendacao.shape[1]}\n")

# 3. Tratar os valores nulos (NaN) substituindo por 0
matriz_knn = matriz_recomendacao.fillna(0)

print("Matriz Final Pronta para o KNN (NaN substituídos por 0):")
display(matriz_knn.head(20))

Bases combinadas com sucesso!
Amostra dos dados combinados (agora com a coluna roteiro):


,id_cliente,id_reserva,roteiro,nota_roteiro
0,CLI002866,RES066382,Via Francigena,9
1,CLI022314,RES060364,Caminho Português,7
2,CLI025723,RES093066,Caminho Primitivo,9
3,CLI010901,RES079044,Via Francigena,8
4,CLI024185,RES064019,Caminho Francês,9


--------------------------------------------------
Matriz de Recomendação criada:
Total de Clientes (Linhas): 27263
Total de Roteiros (Colunas): 5

Matriz Final Pronta para o KNN (NaN substituídos por 0):


roteiro,Caminho Francês,Caminho Português,Caminho Primitivo,Caminho do Norte,Via Francigena
id_cliente,,,,,
CLI000001,8.0,8.0,0.0,0.0,0.0
CLI000002,10.0,0.0,8.0,8.0,9.0
CLI000003,8.0,10.0,7.0,0.0,0.0
CLI000004,7.0,0.0,0.0,8.0,0.0
CLI000005,8.0,8.0,0.0,10.0,0.0
CLI000007,8.0,0.0,0.0,0.0,0.0
CLI000009,0.0,0.0,0.0,0.0,8.0
CLI000010,7.0,0.0,0.0,0.0,9.0
CLI000012,9.0,0.0,7.0,0.0,0.0


In [5]:
# Salva a matriz pronta para ser consumida pelo script de ML
matriz_knn.to_csv("matriz_knn_pronta.csv")